# 01 — EDA and preprocessing

Documents handling of missing `review_content`, `rating`, and `discount_percentage`, currency parsing for ₹ prices, and shared text cleaning from `app/text_clean.py`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from app import config
from app.text_clean import clean_review_text

pd.set_option("display.max_columns", 50)

In [ ]:
df = pd.read_csv(config.RAW_CSV)
print("shape", df.shape)
print(df.dtypes)
key_cols = ["review_content", "rating", "discount_percentage", "discounted_price", "actual_price"]
for c in key_cols:
    miss = df[c].isna().sum() if c in df.columns else None
    empty = (df[c].astype(str).str.strip() == "").sum() if c in df.columns else None
    print(c, "na=", miss, "empty_str=", empty)

## Choices
- Drop rows with empty `review_content` (no usable text).
- Parse `discounted_price` / `actual_price` (strip ₹ and commas) and derive missing discount % when possible.
- Coerce `rating` to float and drop rows that cannot be parsed.
- **No review timestamps** in this file — skip sentiment-over-time; use category and discount instead.

In [ ]:
df["rating_num"] = pd.to_numeric(df["rating"], errors="coerce")
sns.histplot(df["rating_num"].dropna(), bins=20)
plt.title("Rating distribution")
plt.show()

In [ ]:
df["cat0"] = df["category"].astype(str).str.split("|").str[0]
plt.figure(figsize=(10, 4))
order = df.groupby("cat0")["rating_num"].mean().sort_values().index
sns.boxplot(data=df, x="cat0", y="rating_num", order=order)
plt.xticks(rotation=45, ha="right")
plt.title("Rating by top-level category")
plt.tight_layout()
plt.show()

In [ ]:
sample = df["review_content"].dropna().astype(str).head(3)
for raw in sample:
    print("RAW:", raw[:200], "...")
    print("CLEAN:", clean_review_text(raw)[:200], "...\n")

## Save processed table
Run `python scripts/train_and_export.py` from the repo root to write `data/processed/reviews_clean.parquet` with engineered fields, or reuse its `load_and_prepare()` pattern below.

In [ ]:
from scripts.train_and_export import load_and_prepare

clean_df = load_and_prepare()
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
clean_df.to_parquet(config.REVIEWS_CLEAN, index=False)
print("saved", config.REVIEWS_CLEAN, clean_df.shape)